In [0]:
# =====================================================================
# CELL 1: Load the Gold training table and split by TIME
# =====================================================================
import numpy as np
import pandas as pd

# Only rows whose label passed our validity rule (1..1095 days)
df = spark.table("procintel.gold.tender_training").where("is_valid_label").toPandas()

# FEATURES = only what a planner knows BEFORE the tender starts (no leakage)
CAT_FEATURES = ["procurement_group_code", "procurement_method", "review_type",
                "major_sector", "first_notice_type", "env_category"]
NUM_FEATURES = ["log10_amount_usd", "notice_month", "months_since_approval",
                "cpia_public_sector", "wgi_gov_effectiveness",
                "wgi_regulatory_quality", "wgi_control_corruption"]
FEATURES = CAT_FEATURES + NUM_FEATURES
# EXCLUDED on purpose: n_pre_award_notices (re-advertising happens AFTER day one)

# Clean types: categories as text ('UNKNOWN' if missing), numbers as floats
for c in CAT_FEATURES:
    df[c] = df[c].fillna("UNKNOWN").astype(str)
for c in NUM_FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype(float)   # NaN stays NaN; the model handles it

# TEMPORAL SPLIT: sort by notice date, oldest 80% = train, newest 20% = test
df = df.sort_values("first_notice_date").reset_index(drop=True)
cut = int(len(df) * 0.8)
train, test = df.iloc[:cut], df.iloc[cut:]
print(f"train: {len(train)} rows ({train.first_notice_date.min()} → {train.first_notice_date.max()})")
print(f"test:  {len(test)} rows ({test.first_notice_date.min()} → {test.first_notice_date.max()})")

# We model log(days): durations are skewed (many ~150, a few ~900),
# and logs make errors more symmetric. np.log1p(x) = log(1 + x).
y_train, y_test = np.log1p(train.tender_days), np.log1p(test.tender_days)

In [0]:
# =====================================================================
# CELL 2: Train P50 + P90 models, compare to a baseline, log to MLflow
# =====================================================================
import mlflow, sklearn
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor

mlflow.set_registry_uri("databricks-uc")    # register models in Unity Catalog
user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/procintel_tender_duration")   # where runs are recorded

def make_model(loss, quantile=None):
    """A PIPELINE = preprocessing + model in one object, so serving gets both."""
    encoder = ColumnTransformer(
        [("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1,
                                encoded_missing_value=-1), CAT_FEATURES)],
        remainder="passthrough")             # numeric columns pass through unchanged
    params = dict(loss=loss, max_depth=3, learning_rate=0.05, max_iter=300,
                  l2_regularization=1.0, min_samples_leaf=20,   # small data -> strong regularisation
                  categorical_features=[True]*len(CAT_FEATURES) + [False]*len(NUM_FEATURES),
                  random_state=42)
    if quantile is not None:
        params["quantile"] = quantile
    return Pipeline([("encode", encoder), ("gbm", HistGradientBoostingRegressor(**params))])

# A wrapper so ONE served model returns BOTH predictions, in days
class TenderDurationModel(mlflow.pyfunc.PythonModel):
    def __init__(self, p50, p90):
        self.p50, self.p90 = p50, p90
    def predict(self, context, model_input, params=None):
        X = model_input[FEATURES]
        return pd.DataFrame({
            "p50_days": np.expm1(self.p50.predict(X)).round(),   # expm1 undoes log1p
            "p90_days": np.expm1(self.p90.predict(X)).round()},
            index=model_input.index)                             # keep the SAME row labels as the input

with mlflow.start_run(run_name="hgb_p50_p90") as run:
    p50 = make_model("absolute_error").fit(train[FEATURES], y_train)       # median-like
    p90 = make_model("quantile", 0.9).fit(train[FEATURES], y_train)        # 90th percentile
    model = TenderDurationModel(p50, p90)
    pred = model.predict(None, test)

    # BASELINE: median duration per procurement method (from TRAIN only)
    method_median = train.groupby("procurement_method").tender_days.median()
    baseline = test.procurement_method.map(method_median).fillna(train.tender_days.median())

    mae_model    = float(np.mean(np.abs(pred.p50_days.values - test.tender_days.values)))
    mae_baseline = float(np.mean(np.abs(baseline.values - test.tender_days.values)))
    p90_coverage = float(np.mean(test.tender_days.values <= pred.p90_days.values))  # target ≈ 0.90

    mlflow.log_params({"model": "HistGradientBoosting", "max_depth": 3, "learning_rate": 0.05,
                       "max_iter": 300, "n_train": len(train), "n_test": len(test),
                       "split": "temporal 80/20", "features": ",".join(FEATURES)})
    mlflow.log_metrics({"mae_days_model": mae_model, "mae_days_baseline": mae_baseline,
                        "p90_coverage": p90_coverage})

    mlflow.pyfunc.log_model(
        name="model",
        python_model=model,
        input_example=test[FEATURES].head(3),
        signature=infer_signature(test[FEATURES].head(), pred.head()),
        pip_requirements=[f"scikit-learn=={sklearn.__version__}", "pandas", "numpy"],
        registered_model_name="procintel.ml.tender_duration")   # -> Unity Catalog model

print(f"MAE model:    {mae_model:.0f} days")
print(f"MAE baseline: {mae_baseline:.0f} days")
print(f"P90 coverage: {p90_coverage:.0%} (target 90%)")
print("run id:", run.info.run_id)

In [0]:
# =====================================================================
# CELL 3: CHAMPION / CHALLENGER GATE -> promote only if the new model is better
# =====================================================================
# Fair comparison: score the CURRENT champion on THIS run's test set
# (comparing MAEs from different test sets would be apples vs oranges).
from mlflow import MlflowClient
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ServedEntityInput

client, w = MlflowClient(), WorkspaceClient()
MODEL = "procintel.ml.tender_duration"
new_version = max(client.search_model_versions(f"name='{MODEL}'"), key=lambda v: int(v.version)).version

try:
    champion = mlflow.pyfunc.load_model(f"models:/{MODEL}@champion")
    champ_pred = champion.predict(test[FEATURES])
    champ_mae = float(np.mean(np.abs(champ_pred["p50_days"].values - test.tender_days.values)))
except Exception:
    champ_mae = float("inf")                       # no champion yet -> new model wins

print(f"challenger v{new_version} MAE = {mae_model:.1f} | champion MAE on same test = {champ_mae:.1f}")

if mae_model <= champ_mae:
    client.set_registered_model_alias(MODEL, "champion", new_version)
    # Point the live endpoint at the new champion (closed loop: train -> gate -> serve)
    w.serving_endpoints.update_config(
        name="procintel-tender-forecast",
        served_entities=[ServedEntityInput(entity_name=MODEL, entity_version=new_version,
                                           workload_size="Small", scale_to_zero_enabled=True)])
    print(f"🏆 promoted v{new_version} to champion and updated the endpoint")
else:
    print(f"⏸️ kept the current champion; v{new_version} stays registered as a challenger")